# RSNA Knee MRI — Weak Labeling with Qwen2.5-32B

This notebook documents the **Qwen/Qwen2.5-32B-Instruct** zero-shot structured-extraction run that produced the current weak labels for the RSNA Knee Abnormality Detection competition. It replaces the deprecated Qwen3-8B pipeline, whose data card reported a **pooled** ROC AUC (0.851) instead of the competition's **macro** ROC AUC, and whose v1 prompt over-called Effusion (predicted prevalence 0.88 vs 0.60 on the gold studies) and under-called Synovitis (sensitivity 48%).

The model reads each knee MRI report and classifies the 12 target abnormalities (ACL, MCL, menisci, OA compartments, effusion, synovitis, Baker's cyst, contusion, fracture), returning per-target `label`, a calibrated `probability` (stored in the `__confidence` column for schema compatibility), `mentioned` and `negated`.

**Flow:** load data → load Qwen2.5-32B (4-bit) → v2 prompt + strict JSON parser → batched inference → conservative co-occurrence fill → **macro ROC AUC** validation on the 58 ground-truth studies.

> The official full run was executed on a cloud **RTX 4090 (24 GB)** — Qwen2.5-32B at 4-bit needs ~20 GB of VRAM, so it does not fit on one 16 GB Kaggle GPU (it does fit on Kaggle T4 x2 via `device_map="auto"`, or any 24 GB+ GPU). Both output files (`weak_labels_v2_raw.csv` and `weak_labels_v2_filled.csv`) are published in the output dataset; to reproduce the run flip `RUN_GENERATION = True` in the **Setup & configuration** cell (§1) and run on a GPU.


## Output Dataset

The weakly labeled RSNA Knee dataset lives on the same Kaggle dataset page as before (updated in place — the Qwen3-8B file is kept and marked deprecated).

**Qwen2.5-32B v2 weak labels for RSNA Knee Abnormality**

[View the output dataset on Kaggle](https://www.kaggle.com/datasets/laymond/rsna-knee-abnormality-qwen3-8b-weak-labels)

| File | Description |
| ---- | ----------- |
| `weak_labels_v2_filled.csv` | **Production file** — raw output + conservative co-occurrence priors, 4,407 / 4,407 studies `ok` |
| `weak_labels_v2_raw.csv` | Raw Qwen2.5-32B output (no post-processing) |
| `qwen_knee_weak_labels.csv` | Deprecated Qwen3-8B v1 output (kept for reference only) |
| `Qwen2.5-32BGenerator.ipynb` | This generator notebook (also shipped in the dataset) |
| `Qwen3-8BGenerator.ipynb` | Deprecated Qwen3-8B v1 generator (kept for reference only) |


# Model Performance on Labeled Evaluation Set

Model performance is evaluated on the **58 labeled cases** across the 12 target findings (**696 target decisions**), using the production file `weak_labels_v2_filled.csv`. The headline metric is the competition metric **macro ROC AUC**.

> The deprecated Qwen3-8B data card reported an "Overall ROC AUC 0.851" — that was the *pooled* ROC AUC over all 696 decisions concatenated into one curve, which is **not** the competition metric. The macro ROC AUC of those same v1 decisions was ≈0.8295. All numbers below are macro-based / per-target, so they are directly comparable to competition scores.

## Overall Performance

| Metric                       |     Score |
| ---------------------------- | --------: |
| **Macro ROC AUC** (metric)   | **0.8686** |
| **Accuracy**                 | **80.6%** |
| **Precision**                | **67.1%** |
| **Sensitivity (Recall)**     | **85.8%** |
| **F1 Score**                 | **75.3%** |
| Total target decisions       |       696 |
| True positives               |       206 |
| Predicted positives          |       307 |

## Per-Target Performance (classification at the `__label` threshold)

| Target           |  N | TP | TN | FP | FN | Accuracy | Precision | Sensitivity | Specificity |    F1 | Predicted Pos. | Mentioned | Negated | Mean Conf. |
| ---------------- | -: | -: | -: | -: | -: | -------: | --------: | ----------: | ----------: | ----: | -------------: | --------: | ------: | ---------: |
| ACL              | 58 | 24 | 27 |  7 |  0 |   87.9% |    77.4% |     100.0% |      79.4% | 87.3% |         53.4% |    89.7% |  36.2% |      0.50 |
| MCL              | 58 |  9 | 40 |  9 |  0 |   84.5% |    50.0% |     100.0% |      81.6% | 66.7% |         31.0% |    87.9% |  56.9% |      0.28 |
| Medial Meniscus  | 58 | 25 | 23 |  9 |  1 |   82.8% |    73.5% |      96.2% |      71.9% | 83.3% |         58.6% |    94.8% |  34.5% |      0.55 |
| Lateral Meniscus | 58 | 20 | 28 |  7 |  3 |   82.8% |    74.1% |      87.0% |      80.0% | 80.0% |         46.6% |    87.9% |  36.2% |      0.44 |
| Medial OA        | 58 | 14 | 34 |  9 |  1 |   82.8% |    60.9% |      93.3% |      79.1% | 73.7% |         39.7% |    58.6% |  19.0% |      0.38 |
| Lateral OA       | 58 |  9 | 40 |  7 |  2 |   84.5% |    56.2% |      81.8% |      85.1% | 66.7% |         27.6% |    46.6% |  15.5% |      0.28 |
| PF OA            | 58 | 13 | 31 |  6 |  8 |   75.9% |    68.4% |      61.9% |      83.8% | 65.0% |         32.8% |    53.4% |  19.0% |      0.38 |
| Effusion         | 58 | 34 | 10 | 13 |  1 |   75.9% |    72.3% |      97.1% |      43.5% | 82.9% |         81.0% |    96.6% |  10.3% |      0.63 |
| Synovitis        | 58 | 17 | 24 |  7 | 10 |   70.7% |    70.8% |      63.0% |      77.4% | 66.7% |         41.4% |    43.1% |   0.0% |      0.54 |
| Baker's          | 58 | 11 | 40 |  6 |  1 |   87.9% |    64.7% |      91.7% |      87.0% | 75.9% |         29.3% |    50.0% |  19.0% |      0.28 |
| Contusion        | 58 | 17 | 22 | 17 |  2 |   67.2% |    50.0% |      89.5% |      56.4% | 64.2% |         58.6% |    72.4% |  12.1% |      0.53 |
| Fracture         | 58 | 13 | 36 |  4 |  5 |   84.5% |    76.5% |      72.2% |      90.0% | 74.3% |         29.3% |    43.1% |  13.8% |      0.28 |

## Per-Target ROC AUC (filled = production, raw = no co-occurrence fill)

| Target           |  N | Filled AUC | Raw AUC |
| ---------------- | -: | ---------: | ------: |
| ACL              | 58 | 0.981 | 0.981 |
| MCL              | 58 | 0.971 | 0.971 |
| Medial Meniscus  | 58 | 0.897 | 0.897 |
| Lateral Meniscus | 58 | 0.860 | 0.860 |
| Medial OA        | 58 | 0.916 | 0.916 |
| Lateral OA       | 58 | 0.795 | 0.795 |
| PF OA            | 58 | 0.840 | 0.728 |
| Effusion         | 58 | 0.849 | 0.849 |
| Synovitis        | 58 | 0.789 | 0.703 |
| Baker's          | 58 | 0.903 | 0.903 |
| Contusion        | 58 | 0.745 | 0.745 |
| Fracture         | 58 | 0.876 | 0.876 |

| Aggregate | N | Filled AUC | Raw AUC |
| --------- | -: | ---------: | ------: |
| **Macro (competition metric)** | 58 | **0.8686** | **0.8521** |
| Pooled (reference only)        | 696 |     0.8754 |     0.8558 |

## Co-occurrence filling (raw → filled)

Rules apply only to cells the report does **not** mention (`{target}__mentioned == 0`) whose partner has confidence > 0.5; they raise `__confidence` to `max(current, fill)` and never touch `label`/`mentioned`/`negated`. Cells touched on the 4,407-study file: PF OA 385, Synovitis (via Medial OA) 996, Synovitis (via Effusion) 739.

| Apply when | Fill | Empirical P |
| ---------- | ---: | ----------: |
| `PF OA` ← `Medial OA` (silent) | 0.60 | ~0.70 (OA is often multi-compartment) |
| `Synovitis` ← `Medial OA` (silent) | 0.60 | ~0.74 (synovitis is an OA comorbidity) |
| `Synovitis` ← `Effusion` (silent) | 0.55 | ~0.63 |

## Key Observations

* **Best F1:** ACL (**87.3%**); ACL and MCL both reach **100% sensitivity** (no false negatives).
* **Effusion over-calling fixed (v2 prompt):** sensitivity stays high at **97.1%** while specificity recovers to **43.5%** — the deprecated v1 labels were at 30.4% specificity.
* **Synovitis under-calling fixed (v2 prompt):** sensitivity **63.0%** vs 48.1% in v1 after explicitly counting Hoffa fat-pad impingement and related indirect descriptions.
* **Macro ROC AUC:** ≈0.8295 (v1) → **0.8686** (v2 filled); the raw v2 output is already 0.8521.
* **Hardest targets by AUC:** Contusion (0.745), Synovitis (0.789), Lateral OA (0.795).
* **Confidence is now a granular probability** (e.g. Effusion mean 0.63, ACL 0.50) rather than v1's high self-reported certainty — this is what makes the ROC scores meaningful.

Overall the v2 labels are substantially better ranked (macro AUC 0.8686) and much better calibrated on Effusion and Synovitis than the deprecated v1 labels.


## 1. Setup & configuration

In [ ]:
import json
import os
import re
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------
MODEL_ID = "Qwen/Qwen2.5-32B-Instruct"   # 32B - 4-bit needs ~20 GB VRAM
QUANT = "4bit"                            # "4bit" | "8bit" | "none"
BATCH_SIZE = 4                            # reports per generate() call (24 GB GPU: 2-4)
MAX_NEW_TOKENS = 800

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

# Full run: 4,407 studies, ~7 h on a cloud RTX 4090 with batch=4.
# Flip RUN_GENERATION to True to regenerate from train.csv.
RUN_GENERATION = False      # <- set True to reproduce the full run
LIMIT = 8                   # rows to generate (None = all 4,407)

# Data locations -----------------------------------------------------------
# Train reports. In this Kaggle notebook they come from the competition input;
# point this at a local copy when running outside Kaggle.
TRAIN_CSV = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv"

# Outputs (same schema as the official files)
RAW_OUT = "/kaggle/working/weak_labels_v2_raw.csv"
FILL_OUT = "/kaggle/working/weak_labels_v2_filled.csv"

# Official outputs shipped in the dataset (used by the validation cell)
_DATA_DIR = "/kaggle/input/rsna-knee-abnormality-qwen3-8b-weak-labels"
OFFICIAL_RAW = f"{_DATA_DIR}/weak_labels_v2_raw.csv"
OFFICIAL_FILL = f"{_DATA_DIR}/weak_labels_v2_filled.csv"

print(f"model        : {MODEL_ID} ({QUANT})")
print(f"generate     : RUN_GENERATION={RUN_GENERATION}, LIMIT={LIMIT}")
print(f"train.csv    : {TRAIN_CSV}")
print(f"official raw : {os.path.exists(OFFICIAL_RAW)}  (dataset mounted)")
print(f"official fill: {os.path.exists(OFFICIAL_FILL)}  (dataset mounted)")


## 2. Prompt (v2)

The full v2 system prompt is defined in the next cell. Two targets were rewritten after auditing the v1 labels against the 58 gold studies:

* **Effusion** — v1 treated any mention of joint fluid as positive (predicted prevalence 0.88 vs gold 0.60). v2 only counts an effusion when it is a finding of a degree worth reporting ("moderate/large", distending the suprapatellar recess, listed under impression), and asks for a granular probability.
* **Synovitis** — v1 only caught the literal word "synovitis" (sensitivity 48%). v2 explicitly counts the classic indirect descriptions: Hoffa fat-pad impingement / infrapatellar oedema, plica syndrome, synovial thickening/hypertrophy/proliferation.

The output field is now **probability** = the model's calibrated P(present). We keep the CSV column name `confidence` for schema compatibility, but its meaning changed — it is used directly as the soft label and as the ROC score.


In [ ]:
SYSTEM_PROMPT = """You are an expert musculoskeletal radiologist performing structured information extraction from a knee MRI report.

Read the COMPLETE report and classify ALL 12 targets independently.

The report may be written in ANY language (English, Spanish, Chinese, German,
Dutch, French, Italian, Japanese, Korean, Polish, Portuguese, Turkish, ...).
Read it in its original language. Do not translate the task away: terms such as
derrame / épanchement / 积液 / Erguss / effusion / hidartrosis / efüzyon all
mean the same thing.

Use ONLY information explicitly stated in the report. Do not use clinical
history, symptoms, mechanism of injury, age, treatment, or outside knowledge to
invent findings.

For every target determine:

label:
0 = abnormality is absent
1 = abnormality is present

mentioned:
0 = the report gives no information about this specific finding
1 = the report explicitly discusses the relevant structure or finding

negated:
0 = not explicitly negated
1 = explicitly described as absent, normal, intact, preserved, or without the
    relevant pathology

probability [v2]:
Your own calibrated probability (0.00-1.00, two decimals, GRANULAR) that the
finding is truly present. This is NOT the same as confidence in your reading of
the text. Use the full range:
  0.90-1.00 explicit, unambiguous, convincing
  0.70-0.89 strong or clearly stated evidence
  0.45-0.69 equivocal / "possible" / "cannot exclude" / borderline severity
  0.20-0.44 weak or indirect evidence only
  0.00-0.19 almost certainly absent
Do NOT round everything to 0.95 or 0.05. If you are unsure, say so with a
number in the middle of the range. A well-spread probability is more useful
than a confident-looking one.

Rules that apply to all 12 targets:
- Not mentioned at all -> label 0, mentioned 0, negated 0.
  Do NOT interpret silence as a confirmed negative.
- Explicitly normal / intact / preserved -> label 0, mentioned 1, negated 1.
- Explicitly present -> label 1, mentioned 1, negated 0.
- Uncertain wording ("possible", "suspicious for", "cannot exclude",
  "questionable") -> classify on the evidence actually stated and lower the
  probability.
- Be conservative about positives: do not call pathology present merely because
  it could be inferred from another finding.

TARGET DEFINITIONS

ACL
Abnormality specifically involving the anterior cruciate ligament.
label=1 for explicit ACL tear, rupture, discontinuity, abnormal signal, injury,
or other ACL pathology. A partial or complete tear counts. A ruptured
post-surgical graft counts. An INTACT graft does not.
label=0 when the ACL is explicitly normal, intact, preserved, or without
pathology.

MCL
Abnormality specifically involving the medial collateral ligament.
label=1 for explicit MCL tear, rupture, sprain, abnormal signal, thickening,
injury, or other MCL pathology.
label=0 when explicitly normal, intact, or preserved.

MEDIAL MENISCUS
Pathology specifically involving the medial meniscus.
label=1 for explicit tear, rupture, truncation, extrusion, displaced fragment,
abnormal morphology, maceration, or other medial meniscus pathology.
label=0 when explicitly normal or preserved.

LATERAL MENISCUS
Same rules, for the lateral meniscus.

MEDIAL OA
Osteoarthritis of the MEDIAL tibiofemoral compartment.
label=1 for medial compartment cartilage loss / chondrosis / thinning,
marginal osteophytes, subchondral sclerosis, subchondral oedema or cysts in the
medial compartment. Indirect wording counts: "chondrosis", "cartilage loss",
"joint space narrowing" of the medial compartment, "degenerative change" of the
medial compartment.
label=0 when the medial compartment is explicitly normal or preserved.

LATERAL OA
Same rules, for the LATERAL tibiofemoral compartment.

PF OA
Osteoarthritis of the PATellofemoral compartment.
label=1 for patellofemoral cartilage loss / chondrosis, patellar or trochlear
osteophytes, subchondral change in the patellofemoral compartment.
label=0 when explicitly normal or preserved.

EFFUSION  [v2 - rewritten, this target was badly over-called in v1]
Abnormally increased intra-articular joint fluid.
label=1 when the report states a joint effusion or increased intra-articular
fluid of a DEGREE THAT IS BEING REPORTED AS A FINDING.
Important calibration notes:
- Radiologists very often mention a "small", "mild", "trace", "minimal" or
  "physiological" amount of fluid as an incidental remark. Judge these on the
  whole report: if the effusion is small/incidental and not part of the
  impression or the significant findings, it is usually NOT the effusion this
  task is asking about -> lean to label 0 with a probability around 0.30-0.45,
  not 1.
- A "moderate", "large", "marked", "massive" effusion, an effusion that
  distends the suprapatellar recess or bursa, or one that is listed under
  IMPRESSION / CONCLUSION, is a genuine positive -> label 1.
- Fluid in a Baker's cyst, in a paralabral/parameniscal cyst, or in the
  subcutaneous soft tissues is NOT a joint effusion.
- If the report only says "no effusion", label 0, negated 1.
Give a GRANULAR probability rather than 0.95/0.05: an equivocal small effusion
should sit around 0.4-0.6.

SYNOVITIS  [v2 - rewritten, this target was badly under-called in v1]
Inflammation or abnormality of the synovium. This is the target radiologists
describe INDIRECTLY most often, so read for the following equivalents, not
just for the word "synovitis":
- explicit synovitis, synovial inflammation, synovitis/aktivierung,
  sinovitis, 滑膜炎
- synovial thickening, synovial hypertrophy, synovial proliferation,
  villonodular / frond-like synovial change
- HOFFA FAT PAD impingement, Hoffa fat pad oedema / signal change,
  infrapatellar fat pad oedema, 髌下脂肪垫 (this is a classic indirect
  description of synovitis/impingement and COUNTS)
- plica syndrome / medial or lateral plicae when described as thickened,
  inflamed or symptomatic
- suprapatellar bursitis when described with synovial hypertrophy or
  inflammatory change
- friktionssyndrom / impingement syndrome with an inflammatory component
label=1 when any of the above is present with an inflammatory or proliferative
character.
label=0 when the synovium is explicitly normal, or when joint fluid alone is
mentioned without any synovial thickening/ inflammation/ Hoffa oedema.
Note: a bare joint effusion, on its own, is NOT synovitis.

BAKER'S CYST
A cyst in the popliteal fossa (gastrocnemius-semimembranosus bursa).
label=1 for an explicit Baker's cyst, popliteal cyst, or Baker-Zyste / quiste
poplíteo / 腘窝囊肿.
label=0 when explicitly absent or when only a small amount of fluid in a normal
bursa is seen.

CONTUSION
Bone contusion / bone bruise (trabecular microfracture with marrow oedema).
label=1 for explicit bone contusion, bone bruise, marrow oedema of traumatic
character, or "Kontusion" / "contusión" / 骨挫伤.
Do NOT count isolated subchondral oedema that is clearly attributed to
osteoarthritis or to stress reaction without trauma wording.
label=0 when explicitly absent.

FRACTURE
Fracture of any bone about the knee.
label=1 for explicit fracture, cortical disruption, avulsion fracture,
occult fracture, insufficiency fracture, or "Fraktur" / "fractura" / 骨折.
label=0 when explicitly absent.

OUTPUT
Return ONLY a single JSON object, no prose, no markdown fences, with exactly
these 12 keys and for each the four fields "label" (0 or 1), "probability"
(0.00-1.00), "mentioned" (0 or 1), "negated" (0 or 1).
"""

USER_TEMPLATE = """MRI REPORT:
\"\"\"
{report}
\"\"\"

Return the JSON object now."""


## 3. Output parser & normaliser

Qwen2.5-32B tends to return SCREAMING_SNAKE keys (`MEDIAL_MENISCUS`, `BAKER'S_CYST`) where the v1 model used title case. The parser pulls the first JSON object, matches keys loosely (also tolerating `Baker's Cyst` / `Bakers`), and enforces label–mentioned–negated logical consistency (`mentioned=0 ⇒ label=0`; `negated=1 ⇒ label=0`).


In [ ]:
_JSON_RE = re.compile(r"\{.*\}", re.S)


def parse_json(raw: str) -> dict | None:
    """Pull the first JSON object out of the completion."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```[a-zA-Z]*\s*", "", raw)
        raw = re.sub(r"\s*```$", "", raw)
    for cand in (raw, _JSON_RE.search(raw).group(0) if _JSON_RE.search(raw) else ""):
        try:
            obj = json.loads(cand)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return None


_KEY_NORM = {re.sub(r"[^a-z0-9]", "", l.lower()): l for l in LABELS}
_KEY_ALIAS = {
    re.sub(r"[^a-z0-9]", "", k): v
    for k, v in [("Baker's Cyst", "Baker's"), ("Baker cyst", "Baker's"),
                 ("Bakers", "Baker's"), ("bakercyst", "Baker's"),
                 ("Patellofemoral OA", "PF OA"), ("PF OA", "PF OA")]
}


def _match_key(key: str) -> str | None:
    """Map a model-generated JSON key to a canonical LABELS entry.

    Qwen3-8B (v1) emits "Medial Meniscus"; Qwen2.5-32B (v2) normalises keys to
    "MEDIAL_MENISCUS" / "BAKER'S_CYST". Match by stripping non-alphanumerics,
    then fall back to substring matching.
    """
    nk = re.sub(r"[^a-z0-9]", "", (key or "").lower())
    if nk in _KEY_NORM:
        return _KEY_NORM[nk]
    if nk in _KEY_ALIAS:
        return _KEY_ALIAS[nk]
    for k, lab in _KEY_NORM.items():
        if k in nk or nk in k:
            return lab
    return None


def normalise(obj: dict) -> dict:
    """Enforce the output contract, tolerating minor model deviations.

    The v2 prompt returns "probability" = the model's calibrated P(present).
    We keep the CSV column name "confidence" for schema compatibility with the
    deprecated v1 file, but its meaning is now P(present): it is used directly
    as the ROC score and as the soft label for training.
    """
    lookup = {}
    for k, v in (obj or {}).items():
        m = _match_key(k)
        if m and isinstance(v, dict):
            lookup[m] = v
    res = {}
    for lab in LABELS:
        e = lookup.get(lab) or {}
        lab_v = int(e.get("label", 0))
        men = int(e.get("mentioned", 0))
        neg = int(e.get("negated", 0))
        prob = e.get("probability", e.get("confidence", 0.5))
        try:
            prob = float(prob)
        except Exception:
            prob = 0.5
        prob = float(min(max(prob, 0.0), 1.0))
        # logical consistency (same rules as the v1 generator)
        if men == 0:
            neg, lab_v = 0, 0
        if neg == 1:
            lab_v = 0
        res[lab] = {"label": lab_v, "confidence": prob,
                    "mentioned": men, "negated": neg}
    return res


## 4. Load Qwen2.5-32B (4-bit)

Loads the tokenizer and model with automatic dtype/device placement. Requirements for the full run:

* **~20 GB VRAM** at 4-bit — a 24 GB RTX 4090 / RTX 3090, or Kaggle T4 x2 via `device_map="auto"` sharding.
* `bitsandbytes`, `accelerate`, `transformers`, `torch`.

The model is only loaded when `RUN_GENERATION = True` in the configuration cell; everything else in this notebook works without a GPU.


In [ ]:
def _compute_dtype(torch):
    """Pick the best dtype the GPU actually supports.

    Turing (compute capability 7.5 -- RTX 2080 Ti) has NO bf16 units: asking
    for bfloat16 either errors out or silently falls back to a very slow
    simulated path. Ampere (8.0+) and Hopper (9.0) support bf16 natively and
    bf16 is safer against overflow, so prefer it there.
    """
    if not torch.cuda.is_available():
        return torch.float32
    major, minor = torch.cuda.get_device_capability(0)
    name = torch.cuda.get_device_name(0)
    if major >= 8:
        dt = torch.bfloat16
        why = f"sm_{major}{minor} (>=8.0): bf16 supported"
    else:
        dt = torch.float16
        why = f"sm_{major}{minor} (<8.0, no bf16 units): forced fp16"
    print(f"[dtype] {name} -> {why}")
    return dt


def load_model(model_id: str = MODEL_ID, quant: str = QUANT, dtype: str = "auto"):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    if dtype == "auto":
        _dt = _compute_dtype(torch)
    else:
        _dt = {"fp16": torch.float16, "bf16": torch.bfloat16,
               "fp32": torch.float32}[dtype]

    kwargs = dict(device_map="auto", trust_remote_code=True)
    # sdpa works on every GPU we care about and is fine at this batch size.
    if not getattr(load_model, "_warned_attn", False):
        kwargs["attn_implementation"] = "sdpa"
        load_model._warned_attn = True

    if quant == "4bit":
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=_dt,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
        )
    elif quant == "8bit":
        kwargs["load_in_8bit"] = True
    else:
        kwargs["torch_dtype"] = _dt

    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    model.eval()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1024 ** 3
        total = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
        print(f"[model] {model_id} loaded  weights={used:.1f}GB / {total:.1f}GB "
              f"free={total - used:.1f}GB")
    return tok, model


## 5. Inference (batched greedy decoding)

A 32B model is memory-bandwidth bound, so batching is what makes the 4,407-study run feasible: `batch=4` reads the ~20 GB of weights once per four reports instead of once per report (roughly 16 h → 7 h on an RTX 4090). Left padding is used (mandatory for `generate()`), and each output is decoded separately and truncated at its pad token.


In [ ]:
def extract_one(tok, model, report: str, max_new_tokens: int = 1400) -> str:
    import torch

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(report=report[:6000])},
    ]
    text = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tok.pad_token_id,
        )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True)


def extract_batch(tok, model, reports, max_new_tokens: int = MAX_NEW_TOKENS,
                  batch_size: int = BATCH_SIZE):
    """Generate for several reports in ONE forward pass.

    With a 32B model the decode step is memory-bandwidth bound: batch=1 reads
    the ~20GB weights once per report, batch=4 reads them once per FOUR reports.
    That is the difference between ~16h and ~7h for the full 4407-study run.

    Left padding is mandatory for generate(): with right padding the model
    would attend to pad tokens at the END of the sequence. Each output is
    decoded separately and truncated at the first pad token so one short
    completion does not get contaminated by the batch tail.
    """
    import torch

    messages = [
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": USER_TEMPLATE.format(report=r[:6000])}]
        for r in reports
    ]
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
             for m in messages]
    inputs = tok(texts, return_tensors="pt", padding=True,
                 padding_side="left").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tok.pad_token_id,
        )
    L = inputs["input_ids"].shape[1]
    raws = []
    for g in out[:, L:]:
        g = g[g != tok.pad_token_id]        # drop any trailing pad tokens
        raws.append(tok.decode(g, skip_special_tokens=True))
    return raws


## 6. Run over all studies

Flip `RUN_GENERATION = True` in the configuration cell to reproduce the run (`LIMIT` caps the number of studies — set to `None` for all 4,407). It writes:

* `weak_labels_v2_raw.csv` — raw model output (checkpointed every 25 rows; re-running starts from scratch)
* `weak_labels_v2_filled.csv` — raw output + the co-occurrence priors described in the Model Performance section

The official run completed with **4,407 / 4,407 rows `ok` (0 errors)** in ~7 h on a cloud RTX 4090 (batch=4).


In [ ]:
# --------------------------------------------------------------------------
# Generate the raw labels (optional) and apply the co-occurrence fill
# --------------------------------------------------------------------------

# Co-occurrence prior rules. Only UNMENTIONED cells may be touched:
# target mentioned==0 AND partner confidence > 0.5. Fill values are low on
# purpose (they only nudge the soft label) to avoid overfitting.
_FILL_RULES = [
    # (target, partner, fill value)  -- empirical P(target | partner) in parens
    ("PF OA",     "Medial OA", 0.60),   # P(PF OA|MedOA) ~ 0.70
    ("Synovitis", "Medial OA", 0.60),   # P(Syn |MedOA) ~ 0.74
    ("Synovitis", "Effusion",  0.55),   # P(Syn |Eff)   ~ 0.63
]


def process(raw, uid, report):
    """Parse one completion into the output record. Raises on failure."""
    ts = time.time()
    obj = parse_json(raw)
    if obj is None:
        raise ValueError("no JSON object in completion")
    res = normalise(obj)
    rec = {"StudyInstanceUID": uid, "Report": report,
           "qwen_raw": raw[:4000], "status": "ok", "error": "",
           "generation_seconds": round(time.time() - ts, 2)}
    for lab in LABELS:
        r = res[lab]
        rec[f"{lab}__label"] = r["label"]
        rec[f"{lab}__confidence"] = r["confidence"]
        rec[f"{lab}__mentioned"] = r["mentioned"]
        rec[f"{lab}__negated"] = r["negated"]
    return rec


def _prepare_rows(limit):
    if not os.path.exists(TRAIN_CSV):
        raise FileNotFoundError(TRAIN_CSV)
    tr = pd.read_csv(TRAIN_CSV, low_memory=False)
    tr["StudyInstanceUID"] = tr["StudyInstanceUID"].astype(str)
    tr = tr.dropna(subset=["Report"])
    tr["Report"] = tr["Report"].astype(str)
    tr = tr[~tr["StudyInstanceUID"].duplicated()]
    if limit:
        # prefer gold-labelled studies so a small run still scores against the
        # 58 expert studies (train.csv has exactly 58 non-null rows)
        if "ACL" in tr.columns:
            gold_mask = tr["ACL"].notna()
            g, ng = tr[gold_mask], tr[~gold_mask]
            if len(g) >= limit:
                tr = g.head(limit)
            else:
                tr = pd.concat([g, ng.head(limit - len(g))])
        else:
            tr = tr.head(limit)
    return tr.reset_index(drop=True)


def run_generation(limit=LIMIT):
    """Full/reproduce run. Enabled by setting RUN_GENERATION = True above."""
    if not RUN_GENERATION:
        print("RUN_GENERATION = False -> generation skipped. Flip it to True "
              "in the config cell to reproduce the labels from train.csv.")
        return
    if not os.path.exists(TRAIN_CSV):
        print(f"train.csv not found at {TRAIN_CSV} - nothing to generate.")
        return
    print(f"[run] model={MODEL_ID} quant={QUANT} limit={limit} "
          f"batch_size={BATCH_SIZE}")
    tok, model = load_model()

    rows = [dict(r) for _, r in _prepare_rows(limit).iterrows()]
    # length-sorted order so left-padding wastes almost nothing
    order = sorted(range(len(rows)),
                   key=lambda i: len(str(rows[i].get("Report", "") or "")))

    t0 = time.time()
    done = 0
    rows_by_uid = {}
    for s in range(0, len(order), BATCH_SIZE):
        idxs = order[s:s + BATCH_SIZE]
        reports = [str(rows[i].get("Report", "") or "") for i in idxs]
        try:
            raws = extract_batch(tok, model, reports)
        except Exception as e:
            # whole batch failed (e.g. OOM): fall back to one-by-one so a
            # single bad report cannot kill the run
            print(f"  batch of {len(idxs)} failed ({type(e).__name__}: {e}); "
                  f"retrying individually", flush=True)
            raws = []
            for rpt in reports:
                try:
                    raws.append(extract_one(tok, model, rpt))
                except Exception as e2:
                    raws.append(f"__ERROR__ {type(e2).__name__}: {e2}")
        for i, raw in zip(idxs, raws):
            uid = str(rows[i]["StudyInstanceUID"])
            report = str(rows[i].get("Report", "") or "")
            try:
                rec = process(raw, uid, report)
            except Exception as e:  # keep the pipeline alive on one bad report
                rec = {"StudyInstanceUID": uid, "Report": report,
                       "qwen_raw": raw[:4000], "status": "error",
                       "error": f"{type(e).__name__}: {e}"[:300],
                       "generation_seconds": 0.0}
                for lab in LABELS:
                    rec[f"{lab}__label"] = 0
                    rec[f"{lab}__confidence"] = 0.5
                    rec[f"{lab}__mentioned"] = 0
                    rec[f"{lab}__negated"] = 0
            rows_by_uid[uid] = rec
        done += len(idxs)
        if done % 25 < BATCH_SIZE or done == len(order):
            el = time.time() - t0
            print(f"  {done}/{len(order)}  {el/60:.1f} min elapsed  "
                  f"({el/done:.2f} s/report)", flush=True)
            pd.DataFrame(list(rows_by_uid.values())).to_csv(
                RAW_OUT, index=False)   # checkpoint
        if len(rows_by_uid) >= (limit or len(rows)):
            break

    pd.DataFrame(list(rows_by_uid.values())).to_csv(RAW_OUT, index=False)
    print(f"[done] wrote {RAW_OUT}  ({len(rows_by_uid)} rows, "
          f"{(time.time()-t0)/60:.1f} min)")


def apply_fill(d, verbose=True):
    """Conservative co-occurrence priors on UNMENTIONED cells only.

    Only the ``confidence`` column is modified; label/mentioned/negated are
    left untouched. ``confidence = max(confidence, fill)`` so this can only
    raise the soft label of a silent-but-likely finding.
    """
    d = d.copy()
    for target, partner, pval in _FILL_RULES:
        mn = pd.to_numeric(d[f"{target}__mentioned"], errors="coerce") \
            .fillna(0).to_numpy()
        pp = pd.to_numeric(d[f"{partner}__confidence"], errors="coerce") \
            .fillna(0).to_numpy()
        c = np.array(pd.to_numeric(d[f"{target}__confidence"], errors="coerce")
                     .to_numpy(), dtype=np.float64)
        mask = (mn == 0) & (pp > 0.5)
        c[mask] = np.maximum(c[mask], pval)
        d[f"{target}__confidence"] = c
        if verbose:
            print(f"  fill {target:<10} <- {partner:<10} ({pval:.2f}): "
                  f"{int(mask.sum())} cells")
    return d


# Generate raw labels first, then derive the filled production file.
run_generation()

if os.path.exists(RAW_OUT):
    raw_df = pd.read_csv(RAW_OUT, low_memory=False)
    filled_df = apply_fill(raw_df)
    filled_df.to_csv(FILL_OUT, index=False)
    print(f"wrote {FILL_OUT}  ({len(filled_df)} rows)")
else:
    print(f"(no {RAW_OUT} - run_generation() was skipped)")


## 7. Validation vs the 58 gold studies (macro ROC AUC)

The competition metric is **macro ROC AUC** = mean of the 12 per-target ROC AUCs. Pooled ROC AUC (one curve over all 696 decisions) is **not** the metric and is printed below for reference only, clearly labelled.

This cell reads the two official files from the output dataset (no GPU needed) and re-derives the published numbers; freshly generated files are validated automatically too when `RUN_GENERATION = True`.


In [ ]:
# --------------------------------------------------------------------------
# Validation against the 58 gold studies (macro ROC AUC = competition metric)
# --------------------------------------------------------------------------
def validate(out_csv, train_csv, title="VALIDATION"):
    from sklearn.metrics import roc_auc_score

    pred = pd.read_csv(out_csv, low_memory=False)
    tr = pd.read_csv(train_csv, low_memory=False)
    for d in (pred, tr):
        d["StudyInstanceUID"] = d["StudyInstanceUID"].astype(str)

    gold = tr[tr["ACL"].notna()]
    df = gold[["StudyInstanceUID"] + LABELS].merge(
        pred.drop(columns=[c for c in ("Report",) if c in pred.columns]),
        on="StudyInstanceUID", how="inner")
    if len(df) == 0:
        print(f"[validate] no overlap with the gold studies - skip")
        return

    per, yt, ys = [], [], []
    for lab in LABELS:
        g = df[lab].astype(int)
        col = f"{lab}__label"
        if col not in df.columns:
            continue
        p = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
        # v2 semantics: confidence == calibrated P(present), use directly as
        # the score. (v1 confidence was extraction certainty - not comparable.)
        score = pd.to_numeric(
            df[f"{lab}__confidence"], errors="coerce").fillna(0.5).to_numpy()
        if len(np.unique(g)) < 2:
            continue
        a = roc_auc_score(g, score)
        per.append((lab, a, float(g.mean()), float(p.mean())))
        yt.extend(g.tolist())
        ys.extend(score.tolist())

    macro = float(np.mean([a for _, a, _, _ in per]))
    pooled = roc_auc_score(yt, ys) if len(set(yt)) > 1 else float("nan")
    print("\n" + "=" * 62)
    print(title)
    print("=" * 62)
    print(f"{'target':<18}{'AUC':>9}{'gold_prev':>11}{'pred_prev':>11}")
    for lab, a, gp, pp in sorted(per, key=lambda r: r[1]):
        print(f"{lab:<18}{a:>9.4f}{gp:>11.2f}{pp:>11.2f}")
    print("-" * 62)
    print(f"{'MACRO (competition metric)':<18}{macro:>9.4f}")
    print(f"{'POOLED (not the metric)':<18}{pooled:>9.4f}")
    print("=" * 62)


print("Official outputs shipped in the dataset:")
for path, tag in [(OFFICIAL_RAW, "raw labels   "),
                  (OFFICIAL_FILL, "filled labels")]:
    if os.path.exists(path):
        validate(path, TRAIN_CSV, title=f"VALIDATION - {tag.strip()} "
                                        f"(58 gold studies)")
    else:
        print(f"  skip {tag}: {path} not mounted")

if RUN_GENERATION and os.path.exists(RAW_OUT):
    validate(RAW_OUT, TRAIN_CSV, title="VALIDATION - freshly generated RAW")
    if os.path.exists(FILL_OUT):
        validate(FILL_OUT, TRAIN_CSV, title="VALIDATION - freshly generated FILLED")
